# Interpretation, Calibration, and Decision Quality — Project Improved Model Delivery

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb15_interpretation_calibration_project.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Refit the **committed champions** from nb14 on both spines and explain what they learned via permutation importance and partial dependence.
2. Run **error analysis** on each spine — confusion matrix + per-segment metrics for classification; residual analysis + RMSE-by-quintile for regression.
3. Diagnose **probability calibration** on the classification champion using reliability diagrams + Brier score; fix miscalibration with `CalibratedClassifierCV`.
4. Tune a **decision threshold** under an explicit cost matrix (false-negative vs false-positive); run an FN-cost sensitivity sweep.
5. Write the **M3 milestone scaffold** — interpretation findings, error analysis, decision policy (clf) or residual diagnostic (reg) — in stakeholder language.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — Exercise 1 on the classification spine (segment error analysis); Exercise 2 on the regression spine (residual analysis). Complete both before submitting your notebook.

---

## 💼 Why This Matters

nb14 committed two champions: **`LogReg(C=1.0)`** for the State Health Department's screening pipeline (Wisconsin breast cancer) and **`tuned GBM`** for HomeValue Analytics' price-prediction model (California Housing). Both ceremonies closed; both test sets sealed.

But the stakeholders did not stop at *"the model works on a held-out sample"*. The State Health Department's review board wants two more answers:

1. *"Which features drive the model's malignant predictions, and where does the model fail?"* (interpretation + error analysis)
2. *"Are the probabilities the model emits trustworthy for setting a clinical decision threshold?"* (calibration + threshold + cost)

HomeValue Analytics' deployment council wants the parallel:

1. *"Which features drive the model's price predictions, and which neighborhoods or property types does it systematically over- or under-value?"* (interpretation + residual analysis)
2. The probability/threshold question does not apply to regression — but a parallel exists: **conditional reliability of point predictions** (a USD 41K average error means different things on a USD 100K home vs a USD 500K home).

This notebook walks all of these. Sections 1–6 are paired across both spines. Section 7 (calibration + threshold + cost) is **classification-only** because regression has no probability output to calibrate; for regression projects, the parallel decision-quality work is the residual analysis in Section 6.2 plus the per-quintile RMSE diagnostic that follows it.

> **A question that often comes up here:** *"if my project is regression, can I skip Section 7 entirely?"* Yes for the calibration/threshold/cost mechanics, but read the framing — the *concept* of decision quality (translating model output into stakeholder-actionable rules) applies to both. For regression, your decision-quality artifact is the residual-by-quintile chart from Section 6.2 plus a per-quintile RMSE-in-USD report. M3's poster section on "what the model means for decisions" is what each spine produces in its own way.

---

## 1. Setup — Imports, References, Helpers

Same toolkit as nb14, plus three sklearn imports that are specific to interpretation: `permutation_importance`, `PartialDependenceDisplay`, `calibration_curve` / `CalibratedClassifierCV`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score, StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, brier_score_loss,
                              mean_squared_error, mean_absolute_error, r2_score)
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'
GREEN     = '#2ca02c'
RED       = '#d62728'

# --- Week-2 references (same pipelines as nb12-nb14) ---
reference_clf = Pipeline([('scaler', StandardScaler()),
                          ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))])
reference_reg = Pipeline([('scaler', StandardScaler()),
                          ('reg',    LinearRegression())])

def plot_importance_bars(importances, names, ax, errors=None, color=CLF_COLOR,
                         title='Feature importance', top_n=15):
    df = pd.DataFrame({'name': names, 'imp': importances})
    if errors is not None: df['err'] = errors
    df = df.sort_values('imp', ascending=True).tail(top_n)
    if errors is not None:
        ax.barh(df['name'], df['imp'], xerr=df['err'], color=color, edgecolor='black', capsize=4)
    else:
        ax.barh(df['name'], df['imp'], color=color, edgecolor='black')
    ax.set_xlabel('Importance')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

print("✓ Setup, references, helpers loaded")


---

## 2. Load Both Datasets and Refit the Committed Champions

Same 70/30 splits as nb11–nb14 with `random_state=RANDOM_SEED` so the fits are reproducible. The classification champion is `LogReg(C=1.0)`; the regression champion is `GradientBoostingRegressor(lr=0.1, n_estimators=200, max_depth=5)`. Both are refit on the **full training set** (not on a CV fold) — the version that ships.

The test sets remain sealed from nb14's ceremony forward. Everything in this notebook evaluates on the training set via CV or via internal train/val splits.

In [ ]:
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target
# Flip labels so positive class = malignant (sklearn defaults to malignant=0, benign=1).
# This makes "predicted positive" = "flagged for biopsy" — the natural clinical reading.
y_clf = 1 - y_clf
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Refit committed champions from nb14
champion_clf = reference_clf  # LogReg(C=1.0) wrapped in StandardScaler
champion_clf.fit(X_train_clf, y_train_clf)

champion_reg = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5,
                                          random_state=RANDOM_SEED)
champion_reg.fit(X_train_reg, y_train_reg)

print(f'✓ Classification champion refit: {champion_clf.named_steps["clf"].__class__.__name__}(C=1.0)')
print(f'✓ Regression champion refit:     GradientBoostingRegressor(lr=0.1, n=200, depth=5)')
print(f'✓ Both test sets remain SEALED from nb14 — this notebook uses training data only')


---

## 3. Permutation Importance — Paired Across Both Spines

Permutation importance directly answers *"how much does model performance drop when this feature is randomly shuffled?"* — the question stakeholders actually want answered. Computed on the training set (not the test set, which remains sealed) with 10 repeats so each feature gets a mean ± SD.

In [ ]:
# Permutation importance on both spines — 10 repeats, on training set
perm_clf = permutation_importance(champion_clf, X_train_clf, y_train_clf,
                                   scoring='roc_auc', n_repeats=10,
                                   random_state=RANDOM_SEED, n_jobs=-1)
perm_reg = permutation_importance(champion_reg, X_train_reg, y_train_reg,
                                   scoring='r2', n_repeats=10,
                                   random_state=RANDOM_SEED, n_jobs=-1)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
plot_importance_bars(perm_clf.importances_mean, X_train_clf.columns, axes[0],
                     errors=perm_clf.importances_std, color=CLF_COLOR,
                     title='Classification champion (LogReg) — permutation importance ± SD', top_n=15)
plot_importance_bars(perm_reg.importances_mean, X_train_reg.columns, axes[1],
                     errors=perm_reg.importances_std, color=REG_COLOR,
                     title='Regression champion (tuned GBM) — permutation importance ± SD', top_n=8)
fig.suptitle('What each champion is paying attention to — paired permutation importance',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The classification importance bars usually peak at `worst concave points`, `worst perimeter`, `worst radius`, `mean concave points`, and `worst area` — all consistent with the clinical literature on cell-nucleus markers of malignancy. The error bars are small enough that the rank order is reliable.

The regression importance bars peak at `MedInc` (median income) by a wide margin, with `Latitude`, `Longitude`, and `AveOccup` filling out the top four. The geography features (lat/long) earn their place because California has strong location-driven price effects that the GBM captures via its non-linear splits.

For the M3 milestone poster, the rule from nb12 still applies: include a feature in the importance discussion if its mean is at least 2× its SD. Below that threshold, the importance estimate is too noisy to defend.

---

## 4. Partial Dependence — How Predictions Change With a Feature

Permutation importance ranks features by *how much* they matter. PDP shows *in which direction* and *with what shape* they matter. For each feature, PDP averages out the effects of all other features and plots the model's prediction as the focal feature varies across its range.

Three PDPs per spine — the top-3 features by permutation rank. PDP works for both classification (averages predicted probability) and regression (averages predicted target).

In [ ]:
# Top-3 features by permutation rank, per spine
top3_clf_idx = np.argsort(perm_clf.importances_mean)[-3:][::-1]
top3_reg_idx = np.argsort(perm_reg.importances_mean)[-3:][::-1]
top3_clf = [X_train_clf.columns[i] for i in top3_clf_idx]
top3_reg = [X_train_reg.columns[i] for i in top3_reg_idx]
print(f'Top-3 clf features: {top3_clf}')
print(f'Top-3 reg features: {top3_reg}')

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Classification PDP — predicts probability
PartialDependenceDisplay.from_estimator(champion_clf, X_train_clf, features=top3_clf,
                                         ax=axes[0], grid_resolution=30, n_jobs=-1)
for ax, feat in zip(axes[0], top3_clf):
    ax.set_title(f'Clf — PDP for {feat}', fontsize=10, fontweight='bold')

# Regression PDP — predicts target value (in 100K USD units)
PartialDependenceDisplay.from_estimator(champion_reg, X_train_reg, features=top3_reg,
                                         ax=axes[1], grid_resolution=30, n_jobs=-1)
for ax, feat in zip(axes[1], top3_reg):
    ax.set_title(f'Reg — PDP for {feat}', fontsize=10, fontweight='bold')

fig.suptitle('Partial Dependence — direction + shape of each top feature\'s effect on the prediction',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

Classification PDPs show how the predicted probability of malignancy changes as each top feature varies. For `worst concave points` and `worst perimeter`, the curve is monotonically increasing — larger values push the predicted malignancy probability up. The slope is what the LogReg coefficient captures (in the standardized space), but PDP shows it directly in the original units, which is the format clinicians can read.

Regression PDPs show how the predicted house value changes as each top feature varies. For `MedInc` the curve is monotonically increasing and roughly linear above USD 30K median income. For `AveOccup` the curve is downward (more occupants per dwelling → lower predicted value, capturing density effects). For `Latitude` (or `Longitude`) the curve is non-monotonic — California's coastal geography creates pockets of high value at certain latitudes.

> **A question that often comes up here:** *"what about ICE plots? Why not show them?"* ICE (individual conditional expectation) plots overlay a curve for each sample, showing how individual predictions move with the focal feature. PDP is the average. ICE is useful when you suspect strong **interaction effects** (PDP can hide interactions; ICE reveals them as fan-shaped curves). For the M3 milestone, PDP usually suffices; reach for ICE only if the PDP looks suspicious or if the stakeholder asks for individual-level analysis.

**Key takeaway:** Permutation importance + PDP together = the model's "what" and "how" — what it cares about and in which direction. Stakeholders read both as a pair on the M3 poster.

---

## 5. Error Analysis — Where Each Champion Fails

### 5.1 Classification — Confusion Matrix + Per-Segment Metrics

Aggregate accuracy and ROC-AUC hide segment-level failures. The next step is to slice the data along a meaningful dimension and recompute precision / recall per slice. For breast cancer there is no demographic structure in the dataset (anonymized cell measurements), so we slice along the **predicted-probability quartile** — a model-internal segmentation that reveals where the model is most and least confident.

In [ ]:
# 5.1 Out-of-fold predictions for honest error analysis (no test-set touching)
proba_oof_clf = cross_val_predict(champion_clf, X_train_clf, y_train_clf,
                                   cv=cv_clf, method='predict_proba', n_jobs=-1)[:, 1]
pred_oof_clf  = (proba_oof_clf >= 0.5).astype(int)

# Confusion matrix — labels flipped: positive=malignant
cm = confusion_matrix(y_train_clf, pred_oof_clf)
print('=== CLASSIFICATION CHAMPION — Confusion matrix (OOF predictions) ===')
print(pd.DataFrame(cm, index=['Actual Benign', 'Actual Malignant'],
                   columns=['Pred Benign', 'Pred Malignant']))

# Predicted-probability quartiles → per-segment precision/recall
segments = pd.qcut(proba_oof_clf, q=4, labels=['Q1 (lowest p)', 'Q2', 'Q3', 'Q4 (highest p)'])
seg_metrics = []
for seg in segments.categories:
    mask = segments == seg
    if mask.sum() == 0: continue
    seg_metrics.append({
        'segment':   seg,
        'n':         int(mask.sum()),
        'precision': precision_score(y_train_clf[mask], pred_oof_clf[mask], zero_division=0),
        'recall':    recall_score(y_train_clf[mask],    pred_oof_clf[mask], zero_division=0),
        'f1':        f1_score(y_train_clf[mask],        pred_oof_clf[mask], zero_division=0),
    })
seg_df = pd.DataFrame(seg_metrics).set_index('segment')
print('\n=== PER-SEGMENT METRICS (predicted-probability quartiles) ===')
print(seg_df.to_string())

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14, fontweight='bold',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred Benign', 'Pred Malignant'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Actual Benign', 'Actual Malignant'])
ax.set_title('Classification — confusion matrix (OOF)', fontsize=12, fontweight='bold')

ax = axes[1]
im = ax.imshow(seg_df[['precision', 'recall', 'f1']].values, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(['precision', 'recall', 'f1'])
ax.set_yticks(range(len(seg_df))); ax.set_yticklabels(seg_df.index)
for i in range(seg_df.shape[0]):
    for j in range(3):
        v = seg_df.iloc[i, [0,1,2][j]+1]  # column offset
        ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=10)
ax.set_title('Per-segment metrics — by predicted-probability quartile',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


**Reading the output:**

The confusion matrix shows the OOF predictions — every row was classified by a model that had not seen it during fitting (the CV-fold guarantee). On Wisconsin breast cancer, the diagonal cells (true malignant correctly flagged + true benign correctly cleared) typically dominate; the off-diagonals (false negatives + false positives) are small but non-zero.

The per-segment heatmap is what M3 needs. Each row is a slice of the data by predicted probability; each column is a metric. The pattern to look for: **does the model's quality drop in the middle quartiles?** If Q1 (lowest p — confident benign) and Q4 (highest p — confident malignant) both have high precision/recall but Q2 and Q3 are weak, the model is doing well only on the easy cases. That is the segment where additional features or a different model class might help.

For the M3 poster, this heatmap is the **error-analysis figure** for classification projects.

---

### 5.2 Regression — Residual Plot + RMSE-by-Quintile

For regression the error structure is different. The residual (`y_actual - y_pred`) carries continuous information about under- and over-prediction. Two diagnostics:

1. **Residuals-vs-predicted scatter** — should be roughly horizontal around zero. Funnel shapes reveal heteroscedasticity (the model is more accurate on some price ranges than others).
2. **RMSE by predicted-price quintile** — the headline diagnostic for HomeValue's deployment review. *"On the cheapest homes the model is off by USD X; on the most expensive homes by USD Y."*

In [ ]:
pred_oof_reg = cross_val_predict(champion_reg, X_train_reg, y_train_reg, cv=cv_reg, n_jobs=-1)
residuals = y_train_reg - pred_oof_reg

# Quintile-level RMSE in USD
quintiles = pd.qcut(pred_oof_reg, q=5, labels=['Q1 (cheapest)', 'Q2', 'Q3', 'Q4', 'Q5 (most expensive)'])
quintile_rmse = []
for q in quintiles.categories:
    mask = quintiles == q
    rmse = np.sqrt(mean_squared_error(y_train_reg[mask], pred_oof_reg[mask])) * 100_000
    quintile_rmse.append({'quintile': q, 'n': int(mask.sum()), 'RMSE_USD': rmse,
                          'mean_pred_USD': pred_oof_reg[mask].mean() * 100_000})
qdf = pd.DataFrame(quintile_rmse)
print('=== REGRESSION CHAMPION — RMSE by predicted-price quintile ===')
print(qdf.to_string(index=False, formatters={'RMSE_USD': '{:,.0f}'.format,
                                              'mean_pred_USD': '{:,.0f}'.format}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Residuals vs predicted
ax = axes[0]
ax.scatter(pred_oof_reg, residuals, alpha=0.20, s=8, color=REG_COLOR)
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Predicted (100K USD units)')
ax.set_ylabel('Residual (actual - predicted)')
ax.set_title('Residuals vs predicted — heteroscedasticity check', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# RMSE by quintile bar
ax = axes[1]
ax.bar(qdf['quintile'], qdf['RMSE_USD'], color=REG_COLOR, edgecolor='black')
for i, v in enumerate(qdf['RMSE_USD']):
    ax.text(i, v + 1000, f'USD {v:,.0f}', ha='center', fontsize=10)
ax.set_ylabel('RMSE (USD)')
ax.set_title('RMSE by predicted-price quintile — where is the model most reliable?',
             fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=20)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


**Reading the output:**

The residual plot typically shows a funnel pattern on California Housing — small errors on cheap homes, larger errors on expensive ones. This is heteroscedasticity, and it has business consequences: the USD 41K average RMSE the model reports is an aggregate that hides this structure.

The RMSE-by-quintile bar chart makes the heteroscedasticity actionable. *"On the cheapest 20% of homes (under USD 130K predicted) the model is typically off by USD 30K. On the most expensive 20% (above USD 320K predicted) it is typically off by USD 70K."* HomeValue's deployment council can use this to set expectations: low-end pricing is reliable; high-end pricing has wider error bars and might warrant a human-in-the-loop review for properties above some price threshold.

> **A question that often comes up here:** *"is the heteroscedasticity a model problem or a data problem?"* Both. The model is doing its best on the data, but California housing has a **fat right tail** — there are many more homes in the USD 100–300K range than in the USD 500K+ range, so the model has less data to learn the high-end structure. The fix would be either (a) more high-end training data, or (b) a separate model for the upper price tier. Either is a discussion to bring to the next deployment review, not a fix that goes into M3.

**Key takeaway:** The RMSE-by-quintile chart is the regression equivalent of the per-segment classification heatmap. M3 includes one or the other (depending on your project type) as the error-analysis figure on the poster.

---

## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Segment Analysis Findings

**Task:** Look at Section 5.1's per-segment heatmap and the OOF confusion matrix. Write three findings:

1. Which segment has the highest precision? The lowest recall? Why does this make sense given the clinical context?
2. What does the off-diagonal of the confusion matrix translate to in operational terms (false negatives = missed diagnoses, false positives = unnecessary biopsies)?
3. If the State Health Department asked you to add ONE feature to reduce false negatives, which feature category would you suggest based on the importance/PDP evidence?

---

### YOUR FINDINGS HERE:

1. *(your finding on highest-precision / lowest-recall segment)*
2. *(your translation of off-diagonal counts)*
3. *(your one-feature suggestion)*

---

## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Residual Diagnosis Findings

**Task:** Look at Section 5.2's residual scatter and RMSE-by-quintile chart. Write three findings:

1. Does the residual plot show heteroscedasticity (funnel shape) or is it roughly horizontal? What does that mean for HomeValue's pricing accuracy on different segments?
2. What is the RMSE-in-USD ratio between the cheapest quintile and the most expensive quintile? Interpret in stakeholder terms.
3. If HomeValue asked you to recommend a price-threshold above which the deployed model should escalate to a human reviewer, what would you suggest and why?

---

### YOUR FINDINGS HERE:

1. *(your residual-plot interpretation)*
2. *(your RMSE-ratio interpretation in USD)*
3. *(your escalation-threshold recommendation)*

---

## 6. Decision Quality — Calibration + Threshold + Cost (Classification Track Only)

**This section is classification-specific.** Regression projects skip ahead to Section 7. The State Health Department's screening pipeline needs three things from the classification champion:

- **Trustworthy probabilities** — when the model says *"60% chance of malignancy"*, is the actual rate of malignancy ~60% in patients with that prediction?
- **A defensible threshold** — at what predicted probability should the system trigger a confirmatory biopsy?
- **Cost-awareness** — the cost of a false negative (missed cancer) is much higher than the cost of a false positive (unnecessary biopsy); the threshold should reflect that asymmetry.

> **For regression projects (HomeValue Analytics):** The parallel decision-quality artifact is the residual-by-quintile chart from Section 5.2 plus the escalation-threshold recommendation from Exercise 2. There is no probability calibration in regression — the equivalent question is *"are the point predictions reliable enough to act on without human review?"*, which Section 5.2's chart answers directly.

### 6.1 Reliability Diagram — Are the Probabilities Honest?

In [ ]:
# Reliability diagram + Brier on OOF predictions
prob_true, prob_pred = calibration_curve(y_train_clf, proba_oof_clf, n_bins=10, strategy='quantile')
brier = brier_score_loss(y_train_clf, proba_oof_clf)
print(f'Brier score (OOF): {brier:.4f}  (lower = better; 0.0 = perfect, 0.25 = random)')

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(prob_pred, prob_true, marker='o', linewidth=2, color=CLF_COLOR,
        label=f'Champion (Brier = {brier:.4f})')
ax.set_xlabel('Mean predicted probability per bin')
ax.set_ylabel('Observed positive rate per bin')
ax.set_title('Reliability diagram — classification champion', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Reading the output:**

The reliability diagram plots predicted probability bins against observed positive rates. Perfect calibration is the diagonal — when the model says 60%, the true rate is 60%. LogReg with sufficient data is usually well-calibrated out of the box (one of its underrated virtues vs tree-based models). On Wisconsin breast cancer the curve typically sits very close to the diagonal, with Brier score around 0.04 (close to zero = excellent calibration).

If your model class is a Random Forest or Gradient Boosting (which have probability-output mechanisms that are *not* designed for calibration), the curve will deviate visibly from the diagonal — typically S-shaped (overconfident on extreme predictions). The fix is `CalibratedClassifierCV` with `isotonic` or `sigmoid` calibration, fit via internal CV on the training data.

---

### 6.2 Threshold + Cost — Translating Probabilities into Decisions

In [ ]:
# Cost matrix: false negative is much more expensive than false positive on a screening tool
COST_FN = 10.0   # missed malignant case — major clinical cost
COST_FP = 1.0    # unnecessary biopsy follow-up — minor clinical + financial cost

def expected_cost(y_true, y_pred):
    # With flipped labels: positive=malignant; FN=missed malignancy; FP=unnecessary biopsy
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return COST_FN * fn + COST_FP * fp

thresholds = np.linspace(0.05, 0.95, 19)
costs = []
for t in thresholds:
    y_t = (proba_oof_clf >= t).astype(int)
    costs.append(expected_cost(y_train_clf, y_t))
opt_t = thresholds[np.argmin(costs)]
print(f'Optimal threshold (under FN:FP cost ratio = {COST_FN}:{COST_FP}): t = {opt_t:.2f}')
print(f'Cost at t = 0.50 (default):  {costs[np.argmin(np.abs(thresholds - 0.5))]:.0f}')
print(f'Cost at t = {opt_t:.2f} (optimal): {min(costs):.0f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, costs, marker='o', linewidth=2, color=CLF_COLOR)
ax.axvline(opt_t, color=GREEN, linestyle='--', label=f'Optimal threshold: {opt_t:.2f}')
ax.axvline(0.5, color=GREY, linestyle=':', label='Default threshold: 0.50')
ax.set_xlabel('Decision threshold')
ax.set_ylabel(f'Expected cost (FN cost = {COST_FN}, FP cost = {COST_FP})')
ax.set_title('Threshold sweep under explicit cost matrix', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Reading the output:**

The cost-vs-threshold curve is U-shaped. At low thresholds (t = 0.10) the model flags almost every patient as malignant — many false positives, few false negatives — high cost from FP × cost ratio. At high thresholds (t = 0.90) the model flags almost no one — many false negatives — high cost from FN × cost ratio. The minimum is somewhere between, typically around t = 0.20–0.30 when the FN:FP cost ratio is 10:1.

**The optimal threshold is well below the default 0.50.** This is the right answer for a screening tool: when the cost of a missed cancer is ten times the cost of an unnecessary biopsy, you accept more false positives to reduce false negatives. The default `threshold = 0.50` only makes sense when FN and FP costs are equal — which is rarely the case in practice.

For M3 the deliverable is a one-paragraph **decision policy**: *"At a deployed threshold of t = X, the model captures Y% of malignant cases at the cost of Z% false positives. Threshold was chosen by minimizing expected cost under the State Health Department's clinical cost matrix (FN:FP = 10:1). Sensitivity to the cost ratio is reported in the next subsection."*

---

### 6.3 FN-Cost Sensitivity — How Stable Is the Optimal Threshold?

In [ ]:
# Vary the FN:FP cost ratio over a range; recompute optimal threshold each time
fn_cost_grid = [2, 5, 10, 20, 50]
opt_thresholds = []
for fn_cost in fn_cost_grid:
    costs_v = []
    for t in thresholds:
        y_t = (proba_oof_clf >= t).astype(int)
        cm = confusion_matrix(y_train_clf, y_t)
        # Re-derive cost with this FN cost
        cost = fn_cost * cm[1, 0] + 1.0 * cm[0, 1]
        costs_v.append(cost)
    opt_thresholds.append(thresholds[np.argmin(costs_v)])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fn_cost_grid, opt_thresholds, marker='o', linewidth=2, color=CLF_COLOR)
ax.set_xscale('log')
ax.set_xlabel('FN:FP cost ratio (log scale)')
ax.set_ylabel('Optimal threshold')
ax.set_title('Sensitivity of optimal threshold to FN-cost assumption',
             fontsize=12, fontweight='bold')
for x, y in zip(fn_cost_grid, opt_thresholds):
    ax.text(x, y + 0.02, f'{y:.2f}', ha='center', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\n💡 The optimal threshold drops sharply as FN cost rises — a 10:1 vs 50:1 swing matters.')


**Reading the output:**

The optimal threshold drops as the FN:FP cost ratio rises — at 2:1, the optimum is near 0.40; at 50:1, the optimum drops to 0.10 or below. The shape of the curve tells stakeholders **how confident they need to be in the cost ratio** before the threshold becomes load-bearing.

A useful heuristic for M3: report the threshold at three cost ratios — the State Health Department's stated 10:1, plus a "more conservative" 20:1 and a "less conservative" 5:1. The deployment policy can then state: *"Under any reasonable cost ratio between 5:1 and 20:1, the optimal threshold falls between t = 0.15 and t = 0.30. The deployed value t = 0.20 is robust to misspecification of the cost ratio within this range."*

**Key takeaway:** Threshold sensitivity is the diagnostic that turns a single-number policy into a defensible decision. M3 includes both the optimum and the sensitivity sweep.

---

## 7. Project Milestone 3 Scaffold — Problem-Type-Agnostic

The M3 deliverable is the **Improved Model + Interpretation** package. The scaffold below works for both classification and regression projects — fill in whichever blocks apply to your case.

### 7.1 Improved-Model Headline

| Item | Classification project | Regression project |
|---|---|---|
| Champion model | (from nb14 ceremony) | (from nb14 ceremony) |
| CV CI on primary metric | ROC-AUC mean ± half-width | R² mean ± half-width |
| Test-set verdict | INSIDE / ABOVE / BELOW + numeric | INSIDE / ABOVE / BELOW + numeric |
| Lift over Week-2 reference | Δ ROC-AUC vs LogReg(C=1.0) | Δ R² vs OLS, plus USD-RMSE delta |

### 7.2 Interpretation Findings (3 bullets)

1. **What the model pays attention to** — top-3 features by permutation importance, with stakeholder-language gloss.
2. **How those features affect the prediction** — direction + shape from PDP, with one quantitative example (e.g., *"a one-unit increase in `worst concave points` raises the predicted probability of malignancy by ~0.12 on average across the training set"*).
3. **Where the model fails** — the segment with weakest performance (clf) or the price quintile with widest residuals (reg), with a specific recommendation for next-quarter follow-up.

### 7.3 Decision Quality

- **Classification:** Calibration check + reliability diagram + decision-policy paragraph + FN-cost sensitivity sweep.
- **Regression:** Residual-vs-predicted plot + RMSE-by-quintile chart + escalation-threshold recommendation.

### 7.4 Risks + Future Work (3 bullets)

1. The **interpretability gap** between the chosen model and the linear reference (worse for ensembles, better for LogReg / OLS).
2. **Known segments where the model under-performs** (from §5).
3. One **concrete data acquisition or modeling change** that would materially improve next quarter's performance.

---

## 8. Wrap-Up — Key Takeaways

**What landed today:**

1. **Both committed champions are now interpretable.** Permutation importance + PDP give you the *what* and *how* of each model's reasoning, paired across both spines.
2. **Error analysis is structurally different per spine.** Classification → confusion matrix + per-segment metrics. Regression → residual plot + RMSE-by-quintile. Both go on the M3 poster as the "where it fails" figure.
3. **Decision quality is classification-specific.** Calibration + threshold + cost is the workflow for probability outputs; the regression analogue is the residual diagnostic + escalation-threshold recommendation.
4. **The CV-first rule held for the entire notebook.** No cell touched `X_test_clf` or `X_test_reg`. The test-set numbers from nb14's ceremony are the only test-set numbers in your project.

**Bridge to nb16 (Time Series Forecasting):**

nb16 closes the supervised-learning arc with a problem the dual-spine pattern explicitly does NOT cover: **time-indexed data**. Time series demands different splitting (`TimeSeriesSplit` instead of stratified or random `KFold`), different feature engineering (lags, rolling windows), and a different baseline family (naïve, seasonal-naïve, linear-with-lags). The CV-first discipline carries forward; the splitting and features change. Bring nb14's selection-ceremony muscle memory — nb16 ends with another opening of a (separate) locked test window.

> **A question that often comes up at this point:** *"if my project is time-indexed, did I do anything wrong in nb09–nb15?"* If your project's data has a time dimension that you treated as static (random splits, no time-aware CV), the CV-CI you reported is optimistic and the test-set verdict may be misleading. nb16 walks the protocol for a time-indexed project and explicitly contrasts it with the static workflow you just completed. Cross-check your M3 submission against nb16's protocol if you suspect time structure in your data.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (segment analysis, classification) and Exercise 2 (residual diagnosis, regression).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 15 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both permutation importance bar plots render with error bars
- [ ] Both PDP grids render (3 features per spine)
- [ ] Both error-analysis figures render (clf heatmap + reg quintile bars)
- [ ] No cells touched X_test_clf or X_test_reg

### Next Step:

- **Notebook 16** — Time Series Forecasting (Day 16)

---

<center>

**Thank you!**

</center>